# House Price Prediction – Đánh giá Feature Selection
**Mô hình:** SVM (SVR), RandomForest, XGBoost  
**Dataset:** Ames Housing (Kaggle House Prices)

# Cell 1: Import thư viện và Cấu hình môi trường

In [3]:
import numpy as np
import pandas as pd
import warnings
import time
import json
from scipy.stats import skew

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_regression, RFE, SelectFromModel
)
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Đã tải thành công các thư viện!')

Đã tải thành công các thư viện!


# Cell 2: Tải dữ liệu và Xử lý Ngoại lệ (Outliers)

In [4]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Kích thước tập Train ban đầu: {train.shape}')
print(f'Kích thước tập Test ban đầu:  {test.shape}')

train = train.drop(
    train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
)
train = train.reset_index(drop=True)
print(f'Kích thước tập Train sau khi loại outlier: {train.shape}')

y = np.log1p(train['SalePrice'])
train_id = train['Id']
test_id  = test['Id']

train = train.drop(['Id', 'SalePrice'], axis=1)
test  = test.drop(['Id'], axis=1)

n_train = train.shape[0]
df_all = pd.concat([train, test], axis=0, ignore_index=True)
print(f'Kích thước dữ liệu gộp: {df_all.shape}')

Kích thước tập Train ban đầu: (1460, 81)
Kích thước tập Test ban đầu:  (1459, 80)
Kích thước tập Train sau khi loại outlier: (1458, 81)
Kích thước dữ liệu gộp: (2917, 79)


# Cell 3: Xử lý giá trị thiếu (Missing Values)

In [5]:
none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]
for col in none_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna('None')

zero_cols = [
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'GarageCars', 'GarageArea'
]
for col in zero_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna(0)

df_all['LotFrontage'] = df_all.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
df_all['LotFrontage'] = df_all['LotFrontage'].fillna(df_all['LotFrontage'].median())

mode_cols = [
    'MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st',
    'Exterior2nd', 'SaleType', 'Utilities', 'Functional'
]
for col in mode_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna(df_all[col].mode()[0])

print('Tổng số giá trị thiếu còn lại:', df_all.isnull().sum().sum())

Tổng số giá trị thiếu còn lại: 0


# Cell 4: Kỹ thuật Tạo đặc trưng mới (Feature Engineering)

In [6]:
df_all['TotalSF'] = df_all['TotalBsmtSF'] + df_all['1stFlrSF'] + df_all['2ndFlrSF']
df_all['TotalBath'] = (
    df_all['FullBath'] + 0.5 * df_all['HalfBath']
    + df_all['BsmtFullBath'] + 0.5 * df_all['BsmtHalfBath']
)
df_all['HouseAge'] = df_all['YrSold'] - df_all['YearBuilt']
df_all['RemodAge'] = df_all['YrSold'] - df_all['YearRemodAdd']
df_all['TotalPorchSF'] = (
    df_all['OpenPorchSF'] + df_all['EnclosedPorch']
    + df_all['3SsnPorch'] + df_all['ScreenPorch'] + df_all['WoodDeckSF']
)

df_all['IsRemodeled'] = (df_all['YearBuilt'] != df_all['YearRemodAdd']).astype(int)
df_all['HasGarage']   = (df_all['GarageArea'] > 0).astype(int)
df_all['HasBsmt']     = (df_all['TotalBsmtSF'] > 0).astype(int)
df_all['HasFireplace']= (df_all['Fireplaces'] > 0).astype(int)

numeric_feats = df_all.select_dtypes(include=[np.number]).columns
skewed_feats = df_all[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index

for feat in high_skew:
    df_all[feat] = np.log1p(df_all[feat])

df_all = pd.get_dummies(df_all, drop_first=True)

X_train = df_all.iloc[:n_train, :].copy()
X_test  = df_all.iloc[n_train:, :].copy()
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f'Kích thước Ma trận X_train: {X_train.shape}')
print(f'Kích thước Ma trận X_test:  {X_test.shape}')

Kích thước Ma trận X_train: (1458, 267)
Kích thước Ma trận X_test:  (1459, 267)


# Cell 5: Các kỹ thuật Feature Selection

In [7]:
n_features_select = 50 

selector_var = VarianceThreshold(threshold=0.01)
selector_var.fit(X_train)
feat_var = X_train.columns[selector_var.get_support()].tolist()
print(f'VarianceThreshold: {len(feat_var)} features')

selector_kbest = SelectKBest(score_func=f_regression, k=n_features_select)
selector_kbest.fit(X_train, y)
feat_kbest = X_train.columns[selector_kbest.get_support()].tolist()
print(f'SelectKBest:       {len(feat_kbest)} features')

rf_rfe = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
selector_rfe = RFE(estimator=rf_rfe, n_features_to_select=n_features_select, step=0.1)
selector_rfe.fit(X_train, y)
feat_rfe = X_train.columns[selector_rfe.support_].tolist()
print(f'RFE (RF):          {len(feat_rfe)} features')

rf_sfm = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_sfm.fit(X_train, y)
selector_sfm = SelectFromModel(rf_sfm, threshold='median', prefit=True)
feat_sfm = X_train.columns[selector_sfm.get_support()].tolist()
print(f'SelectFromModel:   {len(feat_sfm)} features')

feat_all = X_train.columns.tolist()
print(f'All Features:      {len(feat_all)} features')

feature_sets = {
    'All_Features':        feat_all,
    'VarianceThreshold':   feat_var,
    'SelectKBest_f_reg':   feat_kbest,
    'RFE_RF':              feat_rfe,
    'SelectFromModel_RF':  feat_sfm,
}

VarianceThreshold: 191 features
SelectKBest:       50 features
RFE (RF):          50 features
SelectFromModel:   134 features
All Features:      267 features


# Cell 6: Định nghĩa mô hình cơ bản (SVM, RandomForest, XGBoost)

In [8]:
def get_models():
    models = {
        'SVR': SVR(kernel='rbf', C=10, epsilon=0.05, gamma='scale'),
        'RandomForest': RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        ),
    }
    return models

print('Đã định nghĩa 3 mô hình: SVR (SVM), RandomForest, XGBoost')

Đã định nghĩa 3 mô hình: SVR (SVM), RandomForest, XGBoost


# Cell 7: Đánh giá Cross-Validation (RMSE) – Feature Selection × Model

In [9]:
def rmse_cv(model, X_data, y_data, n_folds=5):
    """Tính RMSE 5-Fold CV. SVR cần StandardScaler."""
    kf = KFold(n_folds, shuffle=True, random_state=42)
    if isinstance(model, SVR):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_data)
        scores = cross_val_score(
            model, X_scaled, y_data,
            scoring='neg_mean_squared_error', cv=kf, n_jobs=-1
        )
    else:
        scores = cross_val_score(
            model, X_data, y_data,
            scoring='neg_mean_squared_error', cv=kf, n_jobs=-1
        )
    return np.sqrt(-scores)

results = []
print('KẾT QUẢ CROSS-VALIDATION (RMSE trên log1p(SalePrice))')

for fs_name, feats in feature_sets.items():
    X_sub = X_train[feats]
    print(f'\n--- Feature Selection: {fs_name} ({len(feats)} features) ---')
    models = get_models()
    for model_name, model in models.items():
        t0 = time.time()
        scores = rmse_cv(model, X_sub, y)
        elapsed = time.time() - t0
        mean_rmse = scores.mean()
        std_rmse  = scores.std()
        print(f'  {model_name:15s}  RMSE = {mean_rmse:.5f} (± {std_rmse:.5f})  [{elapsed:.1f}s]')
        results.append({
            'FeatureSelection': fs_name,
            'N_Features': len(feats),
            'Model': model_name,
            'RMSE_mean': round(mean_rmse, 5),
            'RMSE_std':  round(std_rmse, 5),
            'Time_sec':  round(elapsed, 1),
        })

df_results = pd.DataFrame(results)
print('\n' + '=' * 70)
print('BẢNG TỔNG HỢP (Mean RMSE)')
print('=' * 70)
pivot = df_results.pivot(index='FeatureSelection', columns='Model', values='RMSE_mean')
print(pivot.to_string())

best_row = df_results.loc[df_results['RMSE_mean'].idxmin()]
print(f"\nTỐT NHẤT: {best_row['FeatureSelection']} + {best_row['Model']}  RMSE = {best_row['RMSE_mean']:.5f}")

KẾT QUẢ CROSS-VALIDATION (RMSE trên log1p(SalePrice))

--- Feature Selection: All_Features (267 features) ---
  SVR              RMSE = 0.18527 (± 0.02191)  [2.8s]
  RandomForest     RMSE = 0.13673 (± 0.00794)  [6.0s]
  XGBoost          RMSE = 0.12127 (± 0.00984)  [3.4s]

--- Feature Selection: VarianceThreshold (191 features) ---
  SVR              RMSE = 0.15470 (± 0.01166)  [0.3s]
  RandomForest     RMSE = 0.13678 (± 0.00758)  [3.6s]
  XGBoost          RMSE = 0.12159 (± 0.00802)  [1.7s]

--- Feature Selection: SelectKBest_f_reg (50 features) ---
  SVR              RMSE = 0.14881 (± 0.00696)  [0.2s]
  RandomForest     RMSE = 0.14291 (± 0.00743)  [1.9s]
  XGBoost          RMSE = 0.13745 (± 0.00745)  [1.0s]

--- Feature Selection: RFE_RF (50 features) ---
  SVR              RMSE = 0.14215 (± 0.01223)  [0.1s]
  RandomForest     RMSE = 0.13648 (± 0.00790)  [2.1s]
  XGBoost          RMSE = 0.12373 (± 0.00805)  [1.1s]

--- Feature Selection: SelectFromModel_RF (134 features) ---
  SVR     

# Cell 8: Top đặc trưng quan trọng & Xuất kết quả

In [10]:

print('--- 20 đặc trưng quan trọng (Random Forest) ---')
imp = pd.Series(rf_sfm.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(20).to_string())

df_results.to_csv('feature_selection_results.csv', index=False)
pivot.to_csv('fs_summary_pivot.csv')
imp.head(30).to_csv('top_features_importance.csv')

with open('selected_features.json', 'w') as f:
    json.dump({k: v for k, v in feature_sets.items() if k != 'All_Features'}, f, indent=2)

print('\n-> Đã xuất các file kết quả:')
print('   - feature_selection_results.csv')
print('   - fs_summary_pivot.csv')
print('   - top_features_importance.csv')
print('   - selected_features.json')

--- 20 đặc trưng quan trọng (Random Forest) ---
OverallQual     0.458655
TotalSF         0.315889
CentralAir_Y    0.018515
GrLivArea       0.012799
GarageCars      0.012393
BsmtFinSF1      0.010244
OverallCond     0.010061
TotalBath       0.009927
GarageArea      0.009838
LotArea         0.009790
RemodAge        0.008619
HouseAge        0.006975
GarageYrBlt     0.006965
BsmtUnfSF       0.006523
1stFlrSF        0.006231
YearBuilt       0.005373
YearRemodAdd    0.005030
LotFrontage     0.004784
TotalBsmtSF     0.004298
TotalPorchSF    0.003703

-> Đã xuất các file kết quả:
   - feature_selection_results.csv
   - fs_summary_pivot.csv
   - top_features_importance.csv
   - selected_features.json
